# 07 · Berita di varians (GARCH-X sederhana), target tetap return
`ret[t+1] ~ N(μ, σ²_t)` dengan `log σ²_t = a + b·z_t`, di-fit dengan MLE (slope diberi penalti ridge λ=0.05).
V0 varians konstan · V1 + volatilitas lampau · V2 + `news_count, gpr_mean, nov`. Walk-forward sama dengan 06 (mulai 60, refit tiap 5 hari).
Metrik: **negative log-likelihood** out-of-sample (lebih kecil lebih baik). Didaftarkan sebelum dijalankan.

In [1]:
import numpy as np, pandas as pd
from scipy.optimize import minimize
from utils import *

df, price = load(with_pup=False)
df['lv'] = np.log(df.ret_roll_std5 ** 2 + 1e-8)          # volatilitas lampau
NEWS = ['news_count', 'gpr_mean', 'nov']
SPECS = {'V0': [], 'V1': ['lv'], 'V2': ['lv'] + NEWS}
LAM = 0.05

def fit_vol(Xtr, ytr, Xte):
    m, s = Xtr.mean(), Xtr.std().replace(0, 1)
    Z, Zt = ((Xtr - m) / s).values, ((Xte - m) / s).values
    sd, y_ = ytr.std(), ytr.values / ytr.std()
    def f(th):
        lv = th[1] + Z @ th[2:]
        return np.mean(0.5 * (lv + (y_ - th[0]) ** 2 / np.exp(lv))) + 0.5 * LAM * np.sum(th[2:] ** 2)
    th = minimize(f, np.r_[y_.mean(), 0, np.zeros(Z.shape[1])], method='BFGS').x
    return th[0] * sd, sd ** 2 * np.exp(th[1] + Zt @ th[2:])

def run(d, specs=SPECS, start=60, step=5):
    mu, var = {n: np.full(len(d), np.nan) for n in specs}, {n: np.full(len(d), np.nan) for n in specs}
    for a, b in wf_blocks(len(d), start, step):
        tr, te = d.iloc[a], d.iloc[b]
        for n, c in specs.items():
            mu[n][b], var[n][b] = fit_vol(tr[c], tr.target, te[c])
    y = d.target.values[start:]
    return {n: 0.5 * (np.log(2 * np.pi * var[n][start:]) + (y - mu[n][start:]) ** 2 / var[n][start:]) for n in specs}, \
           {n: np.sqrt(var[n][start:]) for n in specs}, y

nll, sig, y = run(df)
print(f'OOS n={len(y)} hari')
print(pd.Series({n: v.mean() for n, v in nll.items()}, name='NLL medio OOS').round(4).to_string())

OOS n=81 hari
V0   -4.1785
V1   -4.0544
V2   -4.0220


In [2]:
for a, b in (('V0', 'V1'), ('V1', 'V2'), ('V0', 'V2')):
    d = nll[a] - nll[b]                                  # positif = model b lebih baik
    rng, bs = np.random.default_rng(SEED), []
    for _ in range(1000): bs.append(d[block_idx(len(d), 5, rng)].mean())
    s, p = dm_from_diff(d)
    print(f'{a} → {b}: Δ NLL = {d.mean():+.4f}  CI95 [{np.percentile(bs, 2.5):+.4f}, {np.percentile(bs, 97.5):+.4f}]  DM p={p:.3f}')
print('\ncorr(σ OOS, |y|):', {n: round(np.corrcoef(sig[n], np.abs(y))[0, 1], 3) for n in SPECS})

V0 → V1: Δ NLL = -0.1241  CI95 [-0.3342, -0.0030]  DM p=0.123
V1 → V2: Δ NLL = -0.0324  CI95 [-0.1775, +0.0799]  DM p=0.569
V0 → V2: Δ NLL = -0.1565  CI95 [-0.4653, +0.0271]  DM p=0.128

corr(σ OOS, |y|): {'V0': np.float64(-0.162), 'V1': np.float64(-0.185), 'V2': np.float64(-0.147)}


## Kontrol negatif: acak urutan hari kolom berita, ulangi

In [3]:
neg = []
for s in range(20):
    sh = df.copy(); sh[NEWS] = sh[NEWS].sample(frac=1, random_state=s).values
    n_, _, _ = run(sh, {'V1': SPECS['V1'], 'V2': SPECS['V2']})
    neg.append((n_['V1'] - n_['V2']).mean())
real = (nll['V1'] - nll['V2']).mean()
print(f'Δ NLL (V1→V2) asli {real:+.4f} | acak {np.mean(neg):+.4f} ± {np.std(neg):.4f} (maks {np.max(neg):+.4f})')
print('Berita membantu hanya kalau asli jelas di atas distribusi acak.')

Δ NLL (V1→V2) asli -0.0324 | acak -0.0373 ± 0.0557 (maks +0.0512)
Berita membantu hanya kalau asli jelas di atas distribusi acak.
